# Choosing and inspecting resolutions

A finite module is a diagram of vector spaces and maps, and we can describe it by choosing generators and recording the relations they satisfy. Those relations can have relations of their own. A projective resolution keeps following this process, so that each stage explains the kernel left by the preceding one. Here we will work through a small example far enough to see why that extra stage is needed, then inspect what TamerOp stores and draws.

We use rational coefficients and a directly supplied finite diagram. You need familiarity with vector spaces, matrices and finite posets; the [installation instructions](../src/start/install.md) give the packages used below. `OP.Advanced` supplies the constructors and inspection operations needed for this example. The same resolution views accept `OP.resolve(enc)` when `enc` came from a filtration or another encoding construction.

All the algebra takes place in the category of modules on the supplied finite poset. An encoding may recover a module on a larger parameter space, but that fact alone does not identify the two categories' derived groups. The [category discussion](../math_categories.md) explains the hypotheses needed for such comparisons.


In [ ]:
import TamerOp as OP
import CairoMakie
const OA = OP.Advanced
field = OP.CoreModules.QQField()
K = OP.CoreModules.coeff_type(field);


## A module on a diamond

The order is $1<2<4$ and $1<3<4$, with 2 and 3 incomparable. Let $M$ have the one-dimensional space $K=\mathbb Q$ at vertex 1 and zero elsewhere. Its structure maps along the four covers are zero. This is already a finite module, so its encoding can assign each vertex to itself.


In [ ]:
P = OA.FinitePoset(Bool[1 1 1 1; 0 1 0 1; 0 0 1 1; 0 0 0 1])
dims = [1, 0, 0, 0]
M = OA.PModule{K}(P, dims,
    Dict((u,v) => zeros(K, dims[v], dims[u]) for (u,v) in OA.cover_edges(P)); field)
classifier = OP.Encoding.EncodingMap(P, P, [1,2,3,4])
enc = OP.EncodingResult(P, M, classifier)
@assert OA.check_module(M).valid
OP.visualize(M; kind=:module_inspector, vertex=1)


Write $P_i$ for the module supported on the principal upset of vertex $i$: it has a copy of $K$ at every vertex $v\geq i$, with identity maps between nonzero spaces. A map $P_i\to N$ is determined by the image of $1$ at vertex $i$, because the structure maps of $N$ determine its images everywhere above $i$. Thus $\operatorname{Hom}(P_i,N)\cong N_i$. Since exactness of finite modules can be checked at each vertex, this also explains why $P_i$ is projective: maps out of it lift through surjective module maps.

The map $P_1\to M$ keeps the line at 1 and sends every other space to zero. Its kernel has lines at 2, 3 and 4, with both maps to 4 equal to the identity. We need a generator at 2 and another at 3 to cover that kernel. At 4 their images agree, leaving the relation $(1,-1)$ between them. That last relation generates a copy of $P_4$, so the process ends with

$$0\longrightarrow P_4\xrightarrow{d_2}P_2\oplus P_3
\xrightarrow{d_1}P_1\longrightarrow M\longrightarrow0.$$

[![At vertex 4 the resolution is zero to K to K squared to K to zero, with maps the column (1,-1) and the row (1,1); at the other vertices the nonzero map is an identity.](../assets/derived_guides/diamond_resolution.svg)](../assets/derived_guides/diamond_resolution.svg)

The diagram chooses bases in which the two coefficients of $d_2$ are 1 and $-1$. At vertex 4 its image is exactly the kernel of $d_1$; at the other vertices exactness follows from the displayed identity maps. This is a hand calculation of the resolution, including its maps. A relation among earlier generators is often called a *syzygy*; here we can point to the vector that produces one.

Counting the principal projectives in each degree gives the following prediction. These counts are minimal because, at each step, a generator is introduced only where the preceding kernel cannot be generated from lower vertices.

| Degree | Vertex 1 | Vertex 2 | Vertex 3 | Vertex 4 |
| :--- | ---: | ---: | ---: | ---: |
| 0 | 1 | 0 | 0 | 0 |
| 1 | 0 | 1 | 1 | 0 |
| 2 | 0 | 0 | 0 | 1 |


In [ ]:
resolution = OP.resolve(enc; opts=OA.ResolutionOptions(maxlen=2))
table = OA.visual_spec(resolution)
@assert OA.visual_metadata(table).counts == [1 0 0 0; 0 1 1 0; 0 0 0 1]
OA.visual_metadata(table).verification


The default table counts the stored summands. To check whether those counts come from a complete minimal resolution, request `verify=true`. The verification checks that consecutive maps compose to zero, that each kernel equals the preceding image, that the covers use minimal generators, and that the final kernel vanishes. Here the verified counts are the *Betti numbers*: the multiplicities of the principal projectives in the minimal resolution of this finite module.


In [ ]:
verified = OA.visual_spec(resolution; verify=true)
@assert OA.visual_metadata(verified).verification.minimality == :minimal
@assert OA.visual_metadata(verified).verification.completion == :complete
OP.visualize(verified)


## Follow a selected relation into its map

The first summand in degree 1 is supported on a principal upset. Selecting it highlights its source column in $d_1$; the rows describe summands in degree 0. At vertex 4 both degree 1 summands are active, so the stalk map has two columns, just as in the hand calculation.

Setting `support_sheets=true` draws the summands of the selected term in separate panels. This lets us distinguish the two copies that meet at vertex 4. Their horizontal separation is a drawing choice; the panels still describe summands in the resolution, rather than a decomposition of $M$.


In [ ]:
OP.visualize(resolution; kind=:resolution, degree=1, summand=1,
    vertex=4, support_sheets=true, verify=true)


The degree 2 summand records the relation between those two generators. Its stored basis may use a different normalization from our hand calculation, so we check the content of the relation: both coefficients are nonzero and applying $d_1$ to the resulting column gives zero.


In [ ]:
syzygy = OA.visual_spec(resolution; kind=:resolution, degree=2,
    summand=1, vertex=4, verify=true)
D2 = OA.visual_metadata(syzygy).coefficient_matrix
@assert size(D2) == (2,1) && all(!iszero, D2)
D1 = OA.visual_metadata(OA.visual_spec(resolution; kind=:resolution, degree=1)).coefficient_matrix
@assert all(iszero, D1*D2)
OP.visualize(syzygy)


## A cutoff is not a vanishing theorem

If we stop at `maxlen=0`, we retain the projective cover $P_1\to M$ and leave its nonzero kernel unresolved. The cover is minimal, but this prefix does not determine how many later terms there are. Its table therefore contains only degree 0, and verification marks the resolution as truncated.


In [ ]:
prefix = OP.resolve(enc; opts=OA.ResolutionOptions(maxlen=0))
prefix_view = OA.visual_spec(prefix; verify=true)
@assert size(OA.visual_metadata(prefix_view).counts) == (1,4)
@assert OA.visual_metadata(prefix_view).verification.completion == :truncated
OP.visualize(prefix_view)


## Grades are extra mathematical data

The table above needs only the finite order and the summand labels. A picture in a plane needs coordinates as well. Supply *grades*, one coordinate pair for each vertex, whose coordinatewise order agrees with the poset. For this example the four corners of a square give such an order embedding: $u\leq v$ holds exactly when both coordinates of $u$ are at most the corresponding coordinates of $v$.

A dot in the next view records a finite vertex and its multiplicity in the selected resolution degree. Several summands at the same grade are counted together. These remain multiplicities in the finite category; supplying coordinates alone does not establish that they are the Betti numbers of an ambient multigraded free resolution. The [category discussion](../math_categories.md) describes why the distinction matters.


In [ ]:
grades = [(0,0), (1,0), (0,1), (1,1)]
OP.visualize(resolution; kind=:betti_degrees, degree=1, grades, verify=true)


Reversing the list of grades exchanges bottom and top. It therefore fails the order condition, which the request validator reports below. A layout for drawing a Hasse diagram serves another purpose: it places comparable vertices so that arrows are readable, without promising to recover a given coordinatewise parameter order.


In [ ]:
invalid = OA.check_visual_request(resolution; kind=:betti_degrees, grades=reverse(grades))
@assert !invalid.valid
invalid.issues


## The injective direction

There is a dual construction using principal downsets. Write $I_i$ for the module with a line at every vertex $v\leq i$ and identity maps on that support. A map $N\to I_i$ is determined by a linear functional on $N_i$, so $\operatorname{Hom}(N,I_i)\cong N_i^*$. Over a field this functor preserves exact sequences, which makes $I_i$ injective.

Put a line at vertex 4 and zero elsewhere. Its injective resolution embeds it in $I_4$, then uses $I_2\oplus I_3$ and finally $I_1$ to account for the successive cokernels:

$$0\longrightarrow N\longrightarrow I_4\longrightarrow I_2\oplus I_3
\longrightarrow I_1\longrightarrow0.$$

The multiplicities in a minimal injective resolution are called *Bass numbers*. The arrows run toward increasing resolution degree, so selecting a summand in a positive degree marks its target row in the incoming differential.


In [ ]:
dual_dims = [0,0,0,1]
N = OA.PModule{K}(P, dual_dims,
    Dict((u,v) => zeros(K, dual_dims[v], dual_dims[u]) for (u,v) in OA.cover_edges(P)); field)
dual_enc = OP.EncodingResult(P, N, classifier)
injective = OP.resolve(dual_enc; kind=:injective, opts=OA.ResolutionOptions(maxlen=2))
bass = OA.visual_spec(injective; verify=true)
@assert OA.visual_metadata(bass).counts == [0 0 0 1; 0 1 1 0; 1 0 0 0]
OP.visualize(injective; kind=:resolution, degree=1, summand=1, vertex=1, verify=true)


## Inspect a presentation before minimizing it

A projective presentation describes a module as a cokernel: the generators are identified modulo the image of the relation map. A fringe presentation uses a different construction, taking the image of a map from upsets to downsets. Keeping that distinction in view prevents the same matrix from being given two incompatible interpretations.

Here both the generator and relation terms have summands labelled 1 and 2 on a chain. In the displayed relation map, columns are relations and rows are generators. The lower left entry must vanish because there is no nonzero map from the first principal upset to the second. The lower right entry is allowed by the order and happens to be zero in this example. The viewer marks the forced zero with a dagger.

`UpsetPresentation` stores the transpose of this displayed matrix; the figure labels the source columns and target rows after making that conversion.


In [ ]:
Q = OA.FinitePoset(Bool[1 1; 0 1])
U = [OA.principal_upset(Q,1), OA.principal_upset(Q,2)]
A = K[1 2; 0 0]
presentation = OP.IndicatorTypes.UpsetPresentation{K}(Q, U, U, copy(transpose(A)), nothing)
OP.visualize(presentation; degree=1, summand=2, vertex=2, grades=[(0,0),(1,1)])


We can change the bases without changing the presented module. Let $S$ change the relation coordinates and $T$ change the generator coordinates. If the new matrix is $B$, following a vector through the two descriptions gives $AS=TB$, or $B=T^{-1}AS$.

The viewer checks that the supplied changes are invertible and respect the summand grades, then checks this equation. It displays the supplied change without modifying the stored presentation. For a resolution, the same check concerns one selected differential; a change to the whole resolution must transform the adjacent maps and augmentation too.


In [ ]:
S = K[1 1; 0 1]
T = K[1 0; 0 1]
changed = OA.visual_spec(presentation; degree=1, vertex=2,
    basis_change=(; source=S, target=T))
@assert OA.visual_metadata(changed).basis_change.matrix == K[1 3; 0 0]
@assert OA.visual_metadata(changed).represented_dimension == 1
OP.visualize(changed)


The same views accept `OP.IndicatorResolutions.upset_resolution(M)` and `downset_resolution(M)` when their principal summands are recorded. They cannot assign a single birth label to an arbitrary upset that has several minimal vertices. With exact coefficients the equations above are checked exactly; `RealField` instead uses the field's declared tolerances for numerical rank and equality decisions.

For a large example, `matrix_limit` restricts the rows and columns drawn while keeping the full selected matrix in the specification. Support panels use the row limit and retain any explicitly selected summand. The figure states how much is shown, so a cropped picture can still be interpreted as part of a larger computation.

A resolution now gives us a way to turn module maps into a complex of vector spaces. The [derived computation guide](derived.ipynb) uses precisely this diamond to compute Ext classes and their products. The [visualization reference](../src/reference/visualization.md#resolutions-and-presentation-incidence) gives the selection and verification contracts, and the shared [export workflow](../visualization.md#rendering-and-saving) explains how to save a view.
